# Kifaru — poaching detection model

Fine-tunes a YOLO nano model for the aerial surveillance MVP and exports it to
ONNX, which is what the FastAPI service loads in production.

**Run this on Colab with a GPU runtime** (`Runtime → Change runtime type → T4 GPU`).
The deployment server has no GPU, so training here and shipping weights is the
only workable split.

### Why the class set is what it is

A rhino in frame is not poaching. A *person* or *vehicle* near a rhino is — and
closing on one is more so. The model detects objects only; the distance and
approach logic lives above it in `app/geometry.py` and `app/threat.py`.

| id | class | why |
|----|---------|-----------------------------------------|
| 0 | rhino | the asset being protected — also the **ruler** the scorer measures distance with |
| 1 | person | primary threat indicator |
| 2 | vehicle | car/truck/motorcycle, collapsed |
| 3 | weapon | *optional* — escalates an alert, never required to raise one |

Class 3 is trained with **zero instances**, because no usable weapon data
exists — everything public is close-range indoor CCTV of handguns, while ours
would be a rifle ~15px long seen from 80m. `training/DATASETS.md` covers why,
including that the anti-poaching drone literature does not attempt weapon
detection at all and detects people instead. Keeping the slot costs a few
unused parameters and keeps the ids stable for when data does arrive.

> **The trap this avoids:** fine-tuning on rhino-only images rebuilds the
> detection head for a single class, and the model permanently loses COCO's
> `person` / `car` / `truck`. It would then be structurally incapable of
> detecting poaching. Keep people and vehicles in the label set.

> **Why rhino recall matters twice.** Miss the rhino and you do not merely lose
> one box — the scorer loses its ruler, every distance becomes unmeasurable,
> and the frame falls back to a low baseline score. Rhino recall is effectively
> a floor on the whole system.

## 1. Setup

Versions are pinned deliberately — an unpinned reinstall weeks from now can
shift training defaults and make results non-reproducible.

**Run this cell first and read its output.** It prints whether your code is
executing on a Colab VM or on your own machine, which determines everything
else: where files land, whether Drive is needed, and whether cloning the repo
is necessary at all. It only clones when the repo is not already on disk, so a
setup that already has the checkout needs no GitHub token.

Export and model-checking are **not** reimplemented in this notebook; they live
in `training/export_onnx.py` and `training/check_model.py` and are called from
here, so the opset, image size and output-layout checks cannot drift from what
the server actually does.

In [1]:
!pip install -q ultralytics==8.4.118 onnxruntime==1.20.1 onnx onnxslim python-dotenv

import shutil
import subprocess
from pathlib import Path

import ultralytics

ultralytics.checks()

REPO_SLUG = "sip-project-group-15/api"

try:
    import google.colab  # noqa: F401

    ON_COLAB_VM = True
except ImportError:
    ON_COLAB_VM = False

print(
    "runtime:  Colab VM — this filesystem is Google's, not your laptop's"
    if ON_COLAB_VM
    else "runtime:  local — this filesystem IS your laptop's"
)
print("cwd:     ", Path.cwd())


def git(*args) -> subprocess.CompletedProcess:
    return subprocess.run(["git", *args], capture_output=True, text=True)


def find_repo(start: Path) -> Path | None:
    """Find a checkout we are already running inside.

    A local runtime, or an editor that syncs your workspace onto the runtime,
    means the repo is already on disk. Cloning again would shadow the very
    edits you are trying to test, and would need credentials for no reason.
    """
    for candidate in (start, *start.parents):
        if (candidate / "training" / "export_onnx.py").is_file():
            return candidate
    return None


# Everything this notebook calls out to. Checked explicitly, because a clone
# can succeed and still land a commit that predates this code.
REQUIRED = (
    "training/export_onnx.py",
    "training/check_model.py",
    "app/detector.py",
    "app/threat.py",
)


def is_checkout(path: Path) -> bool:
    """Whether this is a git checkout rather than the debris of a failed clone.

    An interrupted or rejected clone leaves its target directory behind. Mere
    existence is not evidence of anything, and treating it as a checkout turns
    a clear failure here into a confusing one several cells later.
    """
    return (path / ".git").is_dir()


def missing_pieces(path: Path) -> list[str]:
    """Expected files absent from a checkout, i.e. work that was never pushed."""
    return [name for name in REQUIRED if not (path / name).exists()]


def read_token() -> tuple[str | None, str]:
    """Read GH_TOKEN from Colab secrets, naming the failure if there is one.

    Only reached when an anonymous clone has already failed. Note that secrets
    are readable *only from the Colab web UI* — through the VS Code extension
    or any other client this times out, so a private repo cannot be cloned
    that way at all.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None, "not running on Colab"

    try:
        return userdata.get("GH_TOKEN"), "found"
    except Exception as error:
        name = type(error).__name__
        if "Timeout" in name:
            return None, (
                "Colab secrets are readable only from the Colab web UI, not "
                "the VS Code extension. Either make the repo public, or run "
                "this notebook at colab.research.google.com"
            )
        if "NotebookAccess" in name:
            return None, (
                "GH_TOKEN exists, but this notebook is not authorised to read "
                "it — open the key icon in the sidebar and enable notebook "
                "access for it"
            )
        if "SecretNotFound" in name:
            return None, "no Colab secret named GH_TOKEN"
        return None, f"could not read GH_TOKEN ({name}: {error})"


def clone(destination: Path) -> None:
    """Clone anonymously; only reach for a token if that is refused.

    Trying the token first would cost a ten-second secret-fetch timeout on
    every run of a public repo, for nothing.
    """
    result = git("clone", "-q", f"https://github.com/{REPO_SLUG}.git", str(destination))
    if not result.returncode:
        print("repo:     cloned ->", destination)
        return

    anonymous_error = result.stderr.strip()
    token, token_status = read_token()
    print("token:   ", token_status)

    if not token:
        raise SystemExit(
            f"git clone failed: {anonymous_error}\n\n"
            f"If {REPO_SLUG} is private, it cannot be cloned without "
            "credentials.\nThe simplest fix for a student project is to make "
            "the repo public.\nOtherwise run this notebook in the Colab web "
            "UI and add a fine-grained\ntoken as a secret named GH_TOKEN "
            "(Contents: Read-only)."
        )

    shutil.rmtree(destination, ignore_errors=True)
    result = git(
        "clone", "-q", f"https://{token}@github.com/{REPO_SLUG}.git", str(destination)
    )
    if result.returncode:
        # git echoes the remote URL on failure and that URL carries the token,
        # so it must never reach the notebook output.
        raise SystemExit(
            "git clone failed even with a token: "
            f"{result.stderr.strip().replace(token, '***')}\n\n"
            f"The token is likely expired or lacks Contents=Read on {REPO_SLUG}."
        )

    print("repo:     cloned with token ->", destination)


REPO_DIR = find_repo(Path.cwd())

if REPO_DIR:
    print("repo:     already here, no clone needed ->", REPO_DIR)
else:
    REPO_DIR = Path("/content/api")

    if REPO_DIR.exists() and not is_checkout(REPO_DIR):
        print("repo:     discarding incomplete checkout at", REPO_DIR)
        shutil.rmtree(REPO_DIR, ignore_errors=True)

    if is_checkout(REPO_DIR):
        result = git("-C", str(REPO_DIR), "pull", "--ff-only", "-q")
        if result.returncode:
            # Do not report success on a failed update: every later cell would
            # then silently run against stale code.
            raise SystemExit(f"git pull failed: {result.stderr.strip()}")
        print("repo:     updated ->", REPO_DIR)
    else:
        clone(REPO_DIR)

missing = missing_pieces(REPO_DIR)
if missing:
    listed = "\n".join(f"  - {name}" for name in missing)
    raise SystemExit(
        f"The checkout at {REPO_DIR} is missing:\n{listed}\n\n"
        "The clone itself worked — this commit simply predates that code, so it\n"
        "has not been pushed yet. From the repo on your machine:\n\n"
        "    git add -A\n"
        '    git commit -m "add distance scoring and training pipeline"\n'
        "    git push\n\n"
        "then re-run this cell. Colab clones from GitHub, so anything that is\n"
        "only on your laptop is invisible here."
    )

# Confirm which commit is about to be trained from; if this is not the work you
# pushed, the export and scoring code here is not the code you just edited.
print("commit:  ", git("-C", str(REPO_DIR), "log", "-1", "--format=%h %s").stdout.strip())

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.2/112.6 GB disk)
runtime:  Colab VM — this filesystem is Google's, not your laptop's
cwd:      /content
repo:     cloned -> /content/api
commit:   cce9453 chore: add a model-checking CLI and consolidate the training notebook


## 2. Mount Drive

Colab wipes `/content` when the session ends (~90 min idle, 12h hard cap). A
long training run that writes only to `/content` loses `best.pt` when the tab
disconnects. Everything below writes to Drive instead.

In [2]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/kifaru")
RUNS_DIR = PROJECT_DIR / "runs"
WEIGHTS_DIR = PROJECT_DIR / "weights"
DATA_DIR = Path("/content/datasets")  # scratch: large, re-downloadable

for directory in (RUNS_DIR, WEIGHTS_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("checkpoints ->", RUNS_DIR)

Mounted at /content/drive
checkpoints -> /content/drive/MyDrive/kifaru/runs


## 3. Config

`imgsz=640` matches what the production ONNX graph will be frozen at — change it
here and the server-side preprocessing must change with it. Aerial targets are
small (SAVMAP's average annotation is 25 × 23 pixels), so lowering it to save
time costs more than it saves.

30 epochs, not 100. This fine-tunes a pretrained backbone on a small dataset,
where most of the gain lands in the first 20 or so; `patience=10` stops it
earlier still on a plateau. A ~25 minute run also means less exposure to Colab
reclaiming the VM mid-training.

Classes come from `app/config.py` so there is a single definition of what id
means what. `weapon` is included and will train with zero instances — see
`training/DATASETS.md` for why no usable weapon data exists. That costs a few
unused parameters and keeps the ids stable for when data does arrive.

In [3]:
import sys

sys.path.insert(0, str(REPO_DIR))
from app import config

CONFIG = {
    "base_model": "yolo26n.pt",  # nano: the only sane size for a 6-core CPU server
    "epochs": 30,                # fine-tuning a pretrained backbone converges fast
    "patience": 10,
    "imgsz": 640,
    "batch": 16,
    "seed": 0,
}

# Read from app/config.py rather than restated here, so the ids the model is
# trained with cannot drift from the ids the server reads them back as.
CLASSES = dict(enumerate(config.CLASS_NAMES))

print(CONFIG)
print(CLASSES)

{'base_model': 'yolo26n.pt',
 'epochs': 100,
 'patience': 20,
 'imgsz': 640,
 'batch': 16,
 'seed': 0}

## 4. Build the training set

A rhino-only dataset trains a model that **cannot raise a single alert**: the
scorer only ever alerts on `person`, `vehicle` or `weapon`, and a rhino alone is
wildlife, not poaching. The first run of this notebook did exactly that.

Worse, the `person` head starts with COCO's pretrained weights (Ultralytics
remaps it by name), and training on images with no people actively *suppresses*
it. Rhino-only data does not merely fail to teach the threat classes — it
unlearns the one that came for free.

So the set is stitched from two sources by `training/build_dataset.py`:

| source | supplies | viewpoint |
|---|---|---|
| `african-wildlife.yaml` | `rhino` | ground level |
| `VisDrone.yaml` | `person`, `vehicle` | aerial |

Both download on demand (~100 MB and ~2.3 GB). Classes are remapped **by name**,
because source ids are arbitrary and a wrong remap poisons the set silently.

`--cap` is what makes this work. VisDrone frames carry hundreds of tiny vehicles
and pedestrians each; merged unrestricted against a few hundred rhino boxes that
is roughly **1000:1**, and a detector trained on it stops predicting rhino
entirely while its headline mAP still looks respectable.

**Read the balance table this prints before starting the run.** If it says
`No training examples for: person, vehicle`, stop — that is the run that cannot
alert. See `training/DATASETS.md` for every source considered.

In [4]:
MERGED = Path("/content/datasets/kifaru-merged")

!python {REPO_DIR}/training/build_dataset.py \
    --source african-wildlife.yaml:rhino=0 \
    --source VisDrone.yaml:pedestrian=1,people=1,car=2,van=2,truck=2,bus=2,motor=2 \
    --output {MERGED} \
    --cap 1500

data_yaml = MERGED / "kifaru.yaml"
print("\n" + data_yaml.read_text())


WARNING ⚠️ Dataset 'african-wildlife.yaml' images not found, missing path '/content/datasets/african-wildlife/images/val'
Unzipping /content/datasets/african-wildlife.zip to /content/datasets/african-wildlife...: 100% ━━━━━━━━━━━━ 3018/3018 2.7Kfiles/s 1.1s0.1s
Dataset download success ✅ (3.2s), saved to /content/datasets

classes: {0: 'buffalo', 1: 'elephant', 2: 'rhino', 3: 'zebra'}
root:    /content/datasets/african-wildlife

rhino is class 2 in the source dataset


## 5. Train

Resumable: if the Colab session drops, re-running picks up from the last Drive
checkpoint rather than starting over.

In [ ]:
from ultralytics import YOLO

RUN_NAME = "kifaru-v2"
last_checkpoint = RUNS_DIR / RUN_NAME / "weights" / "last.pt"

if last_checkpoint.exists():
    print(f"Resuming from {last_checkpoint}")
    model = YOLO(str(last_checkpoint))
    results = model.train(resume=True)
else:
    model = YOLO(CONFIG["base_model"])
    results = model.train(
        data=str(data_yaml),
        epochs=CONFIG["epochs"],
        patience=CONFIG["patience"],
        imgsz=CONFIG["imgsz"],
        batch=CONFIG["batch"],
        seed=CONFIG["seed"],
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        plots=True,
    )

New https://pypi.org/project/ultralytics/8.4.120 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/kifaru-merged/kifaru.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

## 6. Validate

Your current notebook has no metrics at all, so there is no way to tell whether
a retrain helped. Record these numbers for every run.

`mAP50-95` is the headline. **Per-class recall matters more here** — a missed
poacher is far worse than a false alarm a ranger dismisses.

In [ ]:
best_weights = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best_weights))
metrics = model.val(data=str(data_yaml), imgsz=CONFIG["imgsz"])

print(f"\nmAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}\n")

for i, name in CLASSES.items():
    try:
        p, r, ap50, ap = metrics.box.class_result(i)
        print(f"{name:<8} precision={p:.3f} recall={r:.3f} mAP50={ap50:.3f}")
    except (IndexError, KeyError):
        print(f"{name:<8} — no validation examples")

## 7. Export to ONNX

**What training produced is `best.pt`.** That checkpoint is the model — it is
what accumulates the training, what a future run resumes from, and the only
thing a re-export can start from. ONNX is a one-way build artifact frozen out
of it for serving: you cannot train it, and nothing can recover a `.pt` from an
`.onnx`. So `best.pt` goes to Drive and stays there.

The production image serves the ONNX rather than PyTorch. On a 6-core shared
CPU that is the difference between a ~250MB image starting in ~1s and a ~2GB
image starting in 10–20s, with faster inference besides.

The export itself is **not written here** — it is `training/export_onnx.py`,
called below. That script also verifies the result: it loads the graph through
onnxruntime exactly as the server will, and checks the output layout using
`app/detector.py`'s own predicate, so a graph the server cannot parse fails
here instead of silently producing wrong boxes in production.

In [ ]:
destination = WEIGHTS_DIR / "best.onnx"
IMGSZ = CONFIG["imgsz"]

# The checkpoint first: it is the only artifact a future re-export or a resumed
# training run can start from, and /content is wiped when the session ends.
shutil.copy2(best_weights, WEIGHTS_DIR / "best.pt")

!python {REPO_DIR}/training/export_onnx.py "{best_weights}" --destination "{destination}" --imgsz {IMGSZ}

## 8. Rough CPU timing

This measures raw graph throughput on a blank tensor — an upper bound on how
fast the server could go, ignoring video decoding and scoring. Colab's CPU is a
reasonable stand-in; multiply by ~1.5–2× for contention, since the deployment
box shares 6 cores across containers.

Use the per-frame figure to sanity-check the sampling rate. At 30fps, analysing
every frame of a 5-minute clip is 9,000 inferences — far past the 600s nginx
timeout. Sampling ~2 frames/second cuts that 15× and loses nothing
operationally, since poaching activity does not vanish within 500ms.

For the real end-to-end cost on actual footage, run `training/check_model.py`
against a clip from the repo — it times the same detector the API serves, with
decoding and scoring included.

In [ ]:
import time

import numpy as np
import onnxruntime as ort

session = ort.InferenceSession(str(destination), providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
dummy = np.zeros((1, 3, IMGSZ, IMGSZ), dtype=np.float32)

for _ in range(3):  # warm up
    session.run(None, {input_name: dummy})

start = time.perf_counter()
runs = 20
for _ in range(runs):
    session.run(None, {input_name: dummy})
per_frame = (time.perf_counter() - start) / runs

print(f"{per_frame * 1000:.1f} ms/frame on Colab CPU")
print(f"~{per_frame * 2000:.0f} ms/frame estimated on the server\n")
for minutes in (1, 5):
    frames = minutes * 60 * 2  # sampling at 2 fps
    print(f"{minutes}min clip @2fps = {frames} frames ~ {frames * per_frame * 2:.0f}s")

## Next steps

1. Download `best.onnx` from `Drive/MyDrive/kifaru/weights/` into the API repo at
   `models/best.onnx`.
2. Check it before trusting it, from the repo root:

   ```bash
   python training/check_model.py some_clip.mp4 --annotate out/
   ```

   That runs the real detector, tracker and scorer, prints the score breakdown
   per frame, and estimates per-frame cost on the server. Look at `out/` — if
   the boxes are in the wrong place, the export layout is being misread and no
   amount of scorer tuning will help.
3. Commit `models/best.onnx` and push to `main` to deploy.

### Before you have a trained model

You can exercise the entire pipeline today using stock COCO weights, which
already detect `person` and vehicles well:

```bash
python training/export_onnx.py yolo26n.pt --destination models/baseline-coco.onnx
python training/check_model.py clip.mp4 --model models/baseline-coco.onnx \
    --aliases "car=vehicle,truck=vehicle,bus=vehicle,elephant=rhino"
```

COCO has no rhino, so aliasing an animal it *does* know stands in for one and
lets the distance and approach logic be validated on real footage before a
single epoch of training. `models/baseline-*.onnx` is gitignored — it knows
nothing about rhinos and must never reach a deploy.

### Recording results

Keep the per-class precision/recall from step 7 for every run. Detector mAP is
only half the evaluation: the scorer needs its own scenario-level test set —
20–30 clips labelled poaching / benign, including hard negatives like tourists
and rangers near rhinos — measured as alert precision and recall. The two
numbers move independently, and only the second one is what the demo shows.

**Before committing this notebook:** `Edit → Clear all outputs`. Training logs
and embedded plot images make diffs unreadable and bloat the repo.